In [12]:
import sys 
from pathlib import Path

raiz = Path.cwd()
if not (raiz / "src").is_dir():
    raiz = raiz.parent
sys.path.append(str(raiz))

from src import config, data_loader, profiling

print("Raíz del proyecto:", raiz)
print("Carpeta de datos :", config.DIR_DATOS)

Raíz del proyecto: c:\Users\lucas\OneDrive\Escritorio\Proyectazo\Proyecto Taller
Carpeta de datos : C:\Users\lucas\OneDrive\Escritorio\Proyectazo\files


In [ ]:
ANIOS = ["2020","2021","2024"]
CATEGORIA = config.CAT_CIENCIA_TECNOLOGIA 
archivos = data_loader.buscar_archivos(categoria = CATEGORIA , anios=ANIOS)

for a in archivos: 
    print(a["anio"], a["tipo"], a["ruta"].name)


Parte C: Tecnología — Análisis de Brecha Digital con EPH

In [14]:
archivos_hogares=data_loader.filtrar_archivos(archivos,tipo=config.TIPO_HOGAR)   #---------> cual de las dos opciones es la menera correcta? 
archivos_individuos=data_loader.filtrar_archivos(archivos,tipo=config.TIPO_INDIVIDUO)

def verificar_consistencia_columnas(lista):
    
    """
    Verifica que todos los archivos de la lista tengan los mismos nombres de columnas.

    Compara cada archivo contra el primero de la lista, sin importar el orden
    de las columnas. Informa qué columnas faltan o sobran en los que difieren.

    Args:
        lista: archivos devueltos por data_loader.filtrar_archivos.

    Returns:
        True si todos los archivos tienen las mismas columnas, False si alguno difiere.
    """

    consistente = True
    columnas= None

    for archivo in lista:
        colum_actuales = set(data_loader.leer_columnas(archivo["ruta"]))
        if columnas is None:
             columnas = colum_actuales
        else:
            if colum_actuales != columnas:
                print(f"Diferencia encontrada en el archivo: {archivo['ruta'].name}") 
                consistente= False

    return consistente 

print("¿Columnas de hogares consistentes?:", verificar_consistencia_columnas(archivos_hogares))
print("¿Columnas de individuos consistentes?:", verificar_consistencia_columnas(archivos_individuos)) 

¿Columnas de hogares consistentes?: True
¿Columnas de individuos consistentes?: True


In [15]:

ruta_hogar = config.DIR_DATOS / "ciencia_y_tecnologia" / "EPH_usu_hog_tic_t424.txt"
ruta_individuos = config.DIR_DATOS / "ciencia_y_tecnologia" / "EPH_usu_indiv_tic_t424.txt"


def validar_consistencia(ruta_hogar,ruta_individuo):
    filas_hogares=data_loader.leer_filas(ruta_hogar)
    filas_individuos=data_loader.leer_filas(ruta_individuo)

    tupla_hogares=set((hogar.get("CODUSU"),hogar.get("NRO_HOGAR"), hogar.get("AGLOMERADO")) for hogar in filas_hogares)

    individuos_sin_hogares=0

    for indi in filas_individuos:
       
        tuplas_individuos= indi.get("CODUSU"),indi.get("NRO_HOGAR"),indi.get("AGLOMERADO")

        if tuplas_individuos not in tupla_hogares:
            individuos_sin_hogares+=1

    resultado = {
        "total_hogares_unicos": len(tupla_hogares),
        "total_individuos_analizados": len(filas_individuos),
        "individuos_sin_hogar": individuos_sin_hogares,
        "integridad_correcta": individuos_sin_hogares == 0, 
    }

    return resultado
print(validar_consistencia( ruta_hogar,ruta_individuos))


            





{'total_hogares_unicos': 16484, 'total_individuos_analizados': 45192, 'individuos_sin_hogar': 0, 'integridad_correcta': True}


In [16]:
from pathlib import Path
from datetime import datetime
import json
from src import config

filas_hogares=data_loader.leer_filas(ruta_hogar)

def calcular_tecnologia(filas_hogares):
    total_hogares = len(filas_hogares)
    if total_hogares == 0:
        return {}
        
    con_celular = 0
    con_internet = 0
    con_computadora = 0
    
    for hogar in filas_hogares:
        if hogar.get("CELULAR") == "1":
            con_celular += 1
        if hogar.get("INTERNET") == "1":
            con_internet += 1
        if hogar.get("COMPUTADORA") == "1": 
            con_computadora += 1
            
    resultados = {
        "total_hogares": total_hogares,
        "porcentaje_celular": round((con_celular / total_hogares) * 100, 2),
        "porcentaje_internet": round((con_internet / total_hogares) * 100, 2),
        "porcentaje_computadora": round((con_computadora / total_hogares) * 100, 2)
    }
    
    return resultados

datos_calculados = calcular_tecnologia(filas_hogares)

def exportar_informe(datos = datos_calculados, ruta = config.RUTA_REPORTE, si_existe = "timestamp"):
    """Guarda un diccionario en un archivo JSON controlando si ya existe (Modificación 7.2).
    
    Args:
        datos: diccionario serializable a JSON.
        ruta: ruta del archivo (ej: config.DIR_OUTPUT / "tecnologia_informe.json").
        si_existe: "preguntar" (Opción A) o "timestamp" (Opción B).
    """
    ruta = Path(ruta)
  
    ruta.parent.mkdir(parents=True, exist_ok=True)
    
    if ruta.exists():
        if si_existe == "preguntar":
            respuesta = input(f"{ruta.name} ya existe. ¿Sobrescribir? (s/n): ").strip().lower()
            if respuesta != "s":
                print("No se guardó el informe.")
                return None
        elif si_existe == "timestamp": 
            marca = datetime.now().strftime("%Y%m%d_%H%M%S")
            ruta = ruta.with_name(f"{ruta.stem}_{marca}{ruta.suffix}")
        else:
            raise ValueError("si_existe debe ser 'preguntar' o 'timestamp'")
            
    with open(ruta, "w", encoding="utf-8") as archivo: 
        json.dump(datos, archivo, ensure_ascii=False, indent=2)
        
    print(f"Informe guardado exitosamente en: {ruta}")
    return ruta

exportar_informe () 

Informe guardado exitosamente en: C:\Users\lucas\OneDrive\Escritorio\Proyectazo\Proyecto Taller\output\calidad_datos_20261021_235833.json


WindowsPath('C:/Users/lucas/OneDrive/Escritorio/Proyectazo/Proyecto Taller/output/calidad_datos_20261021_235833.json')